In [1]:
import numpy as np

In [6]:
np.random.seed(2*np.random.randint(0, 1_000_000)+1) #Semilla aleatoria

In [27]:
L = 5 #Tamaño de la red

In [39]:
S = np.random.choice([-1, 1], size=(L, L))
S

array([[ 1, -1, -1, -1,  1],
       [-1,  1, -1, -1, -1],
       [-1,  1, -1,  1, -1],
       [-1,  1, -1,  1,  1],
       [ 1, -1, -1, -1,  1]])

In [73]:
def indices_primeros_vecinos(i, j, L, is_periodica=False):
    # funcion que recibe 2 indices i,j y entrega los indices de los 4 primeros vecinos
    # en el orden: [arriba, derecha, abajo, izquierda].
    # Para condiciones no periodicas, los vecinos que no existan tendran indice None

    if(L<=1):
        raise ValueError("El valor de L debe ser mayor a 1.")
    if(j>L-1 or j<0 or i>L-1 or i<0):
        raise ValueError("El valor de los indices debe estar entre 0 y L-1.")

    if(is_periodica):
        # Los booleanos me indicaran si esta o no en la frontera.
        # Empezare con todos en False para solo tener que 
        # verificar los casos de la frontera
        pv = False
        sv = False
        tv = False
        cv = False
        
        if(i==0):
            pv = True
        elif(i==L-1):
            tv = True
        else:
            pass
            
        if(j==0):
            cv = True
        elif(j==L-1):
            sv = True
        else:
            pass

        return [
            (L-1, j) if pv==True else (i-1, j),
            (i, 0) if sv==True else (i, j+1),
            (0, j) if tv==True else (i+1, j),
            (i, L-1) if cv==True else (i, j-1)
        ]
        
    else:
        # Lo booleanos me indicaran si tiene o no vecino en las posiciones
        # Empezare con que tiene veciones en todas lñas posiciones
        # para solo validar las posiciones de la frontera.
        pv = True
        sv = True
        tv = True
        cv = True
        
        if(i==0):
            pv = False
        elif(i==L-1):
            tv = False
        else:
            pass
            
        if(j==0):
            cv = False
        elif(j==L-1):
            sv = False
        else:
            pass

        return [
            (None, None) if pv==False else (i-1, j),
            (None, None) if sv==False else (i, j+1),
            (None, None) if tv==False else (i+1, j),
            (None, None) if cv==False else (i, j-1)
        ]

In [74]:
S

array([[ 1, -1, -1, -1,  1],
       [-1,  1, -1, -1, -1],
       [-1,  1, -1,  1, -1],
       [-1,  1, -1,  1,  1],
       [ 1, -1, -1, -1,  1]])

In [186]:
def indices_segundos_vecinos(i, j, L, is_periodica=False):
    # funcion que recibe 2 indices i,j y entrega los indices de los 4 segundos vecinos
    # en el orden: [arriba_derecha, abajo_derecha, abajo_izquierda, arriba_izquierda].
    # Para condiciones no periodicas, los vecinos que no existan tendran indice None
    
    if(L<=1):
        raise ValueError("El valor de L debe ser mayor a 1.")
    if(j>L-1 or j<0 or i>L-1 or i<0):
        raise ValueError("El valor de los indices debe estar entre 0 y L-1.")

    if(is_periodica):

        pv = False
        sv = False
        tv = False
        cv = False
        
        if(i==0):
            pv = True
        elif(i==L-1):
            tv = True
        else:
            pass
            
        if(j==0):
            cv = True
        elif(j==L-1):
            sv = True
        else:
            pass

        fila_arriba = (L-1) if pv else (i-1)
        fila_abajo = 0 if tv else (i+1)
        col_izq = (L-1) if cv else (j-1)
        col_der = 0 if sv else (j+1)

        return [
            (fila_arriba, col_der),
            (fila_abajo,  col_der), 
            (fila_abajo,  col_izq),
            (fila_arriba, col_izq),
        ]
        
    else:
        ar_i = True
        ar_d = True
        ab_d = True
        ab_i = True
        
        if(i==0):
            ar_i = False
            ar_d = False
        elif(i==L-1):
            ab_d = False
            ab_i = False
        else:
            pass
            
        if(j==0):
            ar_i = False
            ab_i = False
        elif(j==L-1):
            ar_d = False
            ab_d = False
        else:
            pass

        return [
            (None, None) if ar_d==False else (i-1, j+1),
            (None, None) if ab_d==False else (i+1, j+1),
            (None, None) if ab_i==False else (i+1, j-1),
            (None, None) if ar_i==False else (i-1, j-1)
        ]

In [ ]:
# Logica corregida con IA

def _fila_columna_vecinas(i, j, L, is_periodica):
    # Calcula, de forma independiente, a qué fila corresponde moverse
    # "arriba" y "abajo", y a qué columna corresponde moverse
    # "izquierda" y "derecha". Si no es periodica y no existe el vecino
    # en esa direccion, devuelve None para ese valor.
    if(is_periodica):
        fila_arriba = L-1 if i==0 else i-1
        fila_abajo = 0 if i==L-1 else i+1
        col_izq = L-1 if j==0 else j-1
        col_der = 0 if j==L-1 else j+1
    else:
        fila_arriba = None if i==0 else i-1
        fila_abajo = None if i==L-1 else i+1
        col_izq = None if j==0 else j-1
        col_der = None if j==L-1 else j+1

    return fila_arriba, fila_abajo, col_izq, col_der


def indices_primeros_vecinos(i, j, L, is_periodica=False):
    if(j>L-1 or j<0 or i>L-1 or i<0):
        raise ValueError("El valor de los indices debe estar entre 0 y L-1.")

    fila_arriba, fila_abajo, col_izq, col_der = _fila_columna_vecinas(i, j, L, is_periodica)

    return [
        (None, None) if fila_arriba is None else (fila_arriba, j),  # arriba
        (None, None) if col_der is None else (i, col_der),       # derecha
        (None, None) if fila_abajo is None else (fila_abajo, j),    # abajo
        (None, None) if col_izq is None else (i, col_izq),       # izquierda
    ]


def indices_segundos_vecinos(i, j, L, is_periodica=False):
    if(j>L-1 or j<0 or i>L-1 or i<0):
        raise ValueError("El valor de los indices debe estar entre 0 y L-1.")

    fila_arriba, fila_abajo, col_izq, col_der = _fila_columna_vecinas(i, j, L, is_periodica)

    return [
        (None, None) if None in (fila_arriba, col_der) else (fila_arriba, col_der),  # arriba-derecha
        (None, None) if None in (fila_abajo, col_der) else (fila_abajo, col_der),   # abajo-derecha
        (None, None) if None in (fila_abajo, col_izq) else (fila_abajo, col_izq),   # abajo-izquierda
        (None, None) if None in (fila_arriba, col_izq) else (fila_arriba, col_izq),  # arriba-izquierda
    ]
